# 05 - Website Behavior Clustering (optional / interpretive)

Runs 10 unsupervised clustering algorithms on the 30-feature set to surface
behavioral patterns. **Cluster assignments never drive the phishing/safe
decision** - they are for interpretability and threat-actor profiling only.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from src.utils import get_config, setup_logging
from src.features import FeatureEngineer, FeatureSelector
from src.clustering import ClusterTrainer, name_clusters
setup_logging()
cfg = get_config()
train = pd.read_csv(cfg.raw_data_dir / cfg.config.files.train_data)
X = train[cfg.features]; y = train[cfg.target_name]


In [ ]:
# Use engineered+selected features
eng = FeatureEngineer.load(cfg.preprocessing_dir / "feature_engineer.joblib")
sel = __import__("joblib").load(cfg.features_dir / "feature_selector.joblib")
X_eng = eng.transform(X)
X_sel = sel.transform(X_eng)
print(f"Features for clustering: {X_sel.shape}")


## 1. Run all 10 clustering algorithms

In [ ]:
trainer = ClusterTrainer(params_cfg=cfg.params)
report, labels_per_algo = trainer.train_all(X_sel, cfg.clusters_dir)
print(f"Best: {report.best_algorithm} (silhouette={report.best_silhouette:.4f})")


## 2. Algorithm comparison

In [ ]:
rows = [{
    'Algorithm': r.name, 'Clusters': r.n_clusters,
    'Silhouette': r.silhouette, 'Davies-Bouldin': r.davies_bouldin,
    'Calinski-Harabasz': r.calinski_harabasz, 'Note': r.note,
} for r in report.results]
pd.DataFrame(rows).sort_values('Silhouette', ascending=False)


## 3. PCA 2D scatter

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
sample = X_sel.sample(2000, random_state=42)
coords = PCA(n_components=2, random_state=42).fit_transform(sample)
best_model = __import__("joblib").load(
    cfg.clusters_dir / f"{report.best_algorithm.lower()}.joblib")
labels = best_model.predict(sample) if hasattr(best_model, "predict") \
    else getattr(best_model, "labels_", np.zeros(len(sample)))
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(coords[:,0], coords[:,1], c=labels, cmap='tab10', s=20, alpha=0.7)
ax.set_title(f'PCA projection - {report.best_algorithm} clusters')
plt.colorbar(sc); plt.tight_layout(); plt.show()


## 4. Cluster semantic names

In [ ]:
names = name_clusters(report, X_sel, list(X_sel.columns))
for cid, nm in names.items():
    print(f"Cluster {cid}: {nm}")


## Summary

Unsupervised clustering surfaces behavioral segments - typically
"high-risk IP-based deceptive sites", "spoofed-SSL short-lived domains",
"clean established legitimate sites", and "URL-shortener abuse" clusters.
These segments aid threat-actor profiling but are never used as the
classification decision; the supervised model in the next notebook handles
that.
